# 03. PDF/HWP 텍스트 추출 검증
원본 RFP 파일을 직접 읽고 PDF/HWP 추출 성공 여부와 실패 사유를 확인합니다.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser_chunker import read_document

with (PROJECT_ROOT / 'config/default.yaml').open(encoding='utf-8') as f:
    config = yaml.safe_load(f)

metadata_path = Path(config['paths']['metadata'])
raw_documents_path = Path(config['paths']['raw_documents'])
failure_log_path = PROJECT_ROOT / config['paths']['extraction_failures']

metadata = pd.read_csv(metadata_path)
print(f'메타데이터: {len(metadata)}건')
print(f'원본 폴더: {raw_documents_path}')

메타데이터: 100건
원본 폴더: /data/original_data/files


In [2]:
extraction_rows = []

for index, row in metadata.iterrows():
    doc_id = f'doc_{index + 1:03d}'
    file_name = str(row['파일명']).strip()
    file_path = raw_documents_path / file_name
    file_type = file_path.suffix.lower().lstrip('.')

    try:
        text = read_document(file_path)
        if text:
            status = 'success'
            error = ''
        else:
            status = 'empty'
            error = '추출된 텍스트가 비어 있습니다.'
    except Exception as exc:
        text = ''
        status = 'error'
        error = f'{type(exc).__name__}: {exc}'

    extraction_rows.append({
        'doc_id': doc_id,
        'file_name': file_name,
        'file_type': file_type,
        'status': status,
        'text_length': len(text),
        'error': error,
        'text_preview': text[:200].replace('\n', ' '),
    })

extraction_result = pd.DataFrame(extraction_rows)
display(extraction_result.groupby(['file_type', 'status']).size().rename('count').reset_index())

,file_type,status,count
0,hwp,success,96
1,pdf,success,4


In [3]:
successful_samples = (
    extraction_result[extraction_result['status'] == 'success']
    .groupby('file_type', as_index=False)
    .first()
)
display(successful_samples[['doc_id', 'file_name', 'file_type', 'text_length', 'text_preview']])

failures = extraction_result[extraction_result['status'] != 'success'].copy()
failure_log_path.parent.mkdir(parents=True, exist_ok=True)
failures[['doc_id', 'file_name', 'file_type', 'status', 'error']].to_csv(
    failure_log_path, index=False, encoding='utf-8'
)

print(f'추출 성공: {(extraction_result["status"] == "success").sum()}건')
print(f'빈 텍스트/오류: {len(failures)}건')
print(f'실패 로그: {failure_log_path}')
if not failures.empty:
    display(failures[['doc_id', 'file_name', 'file_type', 'status', 'error']])

,doc_id,file_name,file_type,text_length,text_preview
0,doc_001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,hwp,42424,2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 고도화 제안요청서...
1,doc_008,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,230223,-1- 제 안 요 청 서 고려대학교 차세대 포털·학사 정보...


추출 성공: 100건
빈 텍스트/오류: 0건
실패 로그: /data/processed/extraction_failures.csv


# 14. 표준 metadata.csv 검증
원본 CSV를 문서당 1행의 표준 metadata.csv로 변환한 결과와, 실제 청크 metadata의 일치 여부를 확인합니다.

In [4]:
import json

from scripts.build_metadata import OUTPUT_COLUMNS, REQUIRED_FIELDS

normalized_metadata_path = Path(config['paths']['normalized_metadata'])
chunks_path = Path(config['paths']['chunks'])
standard_metadata = pd.read_csv(
    normalized_metadata_path, dtype=str, encoding='utf-8'
).fillna('')

missing_required = {
    field: int(standard_metadata[field].str.strip().eq('').sum())
    for field in REQUIRED_FIELDS
}

print(f'표준 메타데이터 행 수: {len(standard_metadata)}')
print(f'doc_id 중복: {standard_metadata["doc_id"].duplicated().sum()}건')
print('필수값 누락:', missing_required)
display(standard_metadata.head(5)[OUTPUT_COLUMNS])

표준 메타데이터 행 수: 100
doc_id 중복: 0건
필수값 누락: {'doc_id': 0, 'file_name': 0, 'project_name': 0, 'title': 0, 'agency': 0, 'document_type': 0, 'source_path': 0}


,doc_id,file_name,project_name,title,agency,document_type,file_type,page,source_path,source_exists,notice_number,notice_round,budget,published_at,bid_start_at,bid_end_at,summary
0,doc_001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학,hwp,hwp,,/data/original_data/files/한영대학_한영대학교 특성화 맞춤형 교...,True,20241001798,0.0,130000000.0,2024-10-04 13:51:23,,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...
1,doc_002,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,한국연구재단,hwp,hwp,,/data/original_data/files/한국연구재단_2024년 대학산학협력활...,True,20241002912,0.0,129300000.0,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...
2,doc_003,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,EIP3.0 고압가스 안전관리 시스템 구축 용역,EIP3.0 고압가스 안전관리 시스템 구축 용역,한국생산기술연구원,hwp,hwp,,/data/original_data/files/한국생산기술연구원_EIP3.0 고압가...,True,20240827859,0.0,40000000.0,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...
3,doc_004,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,도시계획위원회 통합관리시스템 구축용역,도시계획위원회 통합관리시스템 구축용역,인천광역시,hwp,hwp,,/data/original_data/files/인천광역시_도시계획위원회 통합관리시스...,True,20240430918,0.0,150000000.0,2024-04-18 16:26:32,2024-05-02 10:00:00,2024-05-09 16:00:00,- 사업명: 도시계획위원회 통합관리시스템 구축 용역\n- 용역개요: 도시계획위원회와...
4,doc_005,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,hwp,hwp,,/data/original_data/files/경상북도 봉화군_봉화군 재난통합관리시...,True,20240430896,0.0,900000000.0,2024-04-18 16:33:28,2024-04-26 09:00:00,2024-04-30 17:00:00,- 사업명: 봉화군 재난통합관리시스템 고도화 사업\n- 사업개요: 공동수급(공동이행...


In [5]:
metadata_by_doc_id = {row['doc_id']: row for row in standard_metadata.to_dict(orient='records')}
mismatches = []
chunk_count = 0

with chunks_path.open(encoding='utf-8') as file:
    for line in file:
        if not line.strip():
            continue

        chunk = json.loads(line)
        chunk_count += 1
        expected = metadata_by_doc_id.get(chunk['doc_id'])

        if expected is None:
            mismatches.append({'chunk_id': chunk['chunk_id'], 'field': 'doc_id', 'reason': 'metadata.csv에 없음'})
            continue

        for field in OUTPUT_COLUMNS:
            actual_value = str(chunk['metadata'].get(field, ''))
            expected_value = str(expected[field])
            if actual_value != expected_value:
                mismatches.append({
                    'chunk_id': chunk['chunk_id'],
                    'field': field,
                    'actual': actual_value,
                    'expected': expected_value,
                })
                break

print(f'검증 청크 수: {chunk_count}')
print(f'청크-메타데이터 불일치: {len(mismatches)}건')
if mismatches:
    display(pd.DataFrame(mismatches).head(10))
else:
    print('모든 청크 metadata가 표준 metadata.csv와 일치합니다.')

검증 청크 수: 10222
청크-메타데이터 불일치: 0건
모든 청크 metadata가 표준 metadata.csv와 일치합니다.


## 27. 표·이미지 기반 OCR 및 추가 추출 필요성 점검

PDF와 HWP에 포함된 표·이미지 수를 확인해 OCR 또는 표 추출을 추가로 검토할 문서를 찾습니다.

- PDF 표 수는 PyMuPDF가 탐지한 표 후보 수입니다.
- HWP 표 수는 HWP 본문 스트림의 표 레코드 수입니다.
- 이미지가 있다고 바로 OCR을 적용하지는 않습니다. 이미지 안에 글자가 실제로 있는지 대표 문서를 추가 확인한 뒤 결정합니다.

In [6]:
import shutil

from src.parser_chunker import inspect_document_visual_content

visual_reports = [
    inspect_document_visual_content(path)
    for path in sorted(raw_documents_path.iterdir())
    if path.suffix.lower() in {'.pdf', '.hwp'}
]
visual_result = pd.DataFrame(visual_reports)

visual_display = visual_result[
    [
        'file_name',
        'document_type',
        'page_count',
        'table_count',
        'table_count_basis',
        'image_count',
        'empty_page_count',
        'visual_content_recommendation',
        'ocr_recommendation',
    ]
].copy()
visual_display.columns = [
    '파일명',
    '문서 형식',
    '페이지 수',
    '표 수',
    '표 수 기준',
    '이미지 수',
    '텍스트 미추출 페이지 수',
    '표·이미지 추가 검토',
    'OCR 검토',
]
visual_display['문서 형식'] = visual_display['문서 형식'].map({'pdf': 'PDF', 'hwp': 'HWP'})
visual_display['표 수 기준'] = visual_display['표 수 기준'].map({
    'pdf_table_candidate': 'PDF 표 후보',
    'hwp_table_record': 'HWP 표 레코드',
})
visual_display['표·이미지 추가 검토'] = visual_display['표·이미지 추가 검토'].map({
    'not_required': '추가 검토 불필요',
    'table_extraction_review_required': '표 추출 검토 필요',
    'image_ocr_review_required': '이미지 OCR 검토 필요',
    'table_and_image_review_required': '표 추출·이미지 OCR 검토 필요',
})
visual_display['OCR 검토'] = visual_display['OCR 검토'].map({
    'not_checked': '빈 페이지 기준 미점검',
    'not_required': 'OCR 불필요',
    'review_required': '추가 검토 필요',
    'ocr_required': 'OCR 적용 검토 필요',
})
display(visual_display)

print(f'확인한 문서 수: {len(visual_result)}개')
print(f'PDF 수: {(visual_result["document_type"] == "pdf").sum()}개')
print(f'HWP 수: {(visual_result["document_type"] == "hwp").sum()}개')
print(f'Tesseract 설치 여부: {"설치됨" if shutil.which("tesseract") else "설치되지 않음"}')

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


MuPDF error: syntax error: invalid key in dict



MuPDF error: syntax error: invalid key in dict



,파일명,문서 형식,페이지 수,표 수,표 수 기준,이미지 수,텍스트 미추출 페이지 수,표·이미지 추가 검토,OCR 검토
0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp,HWP,NaN,226,HWP 표 레코드,9,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,HWP,NaN,107,HWP 표 레코드,17,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
2,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,HWP,NaN,151,HWP 표 레코드,6,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
3,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp,HWP,NaN,95,HWP 표 레코드,6,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
4,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,HWP,NaN,91,HWP 표 레코드,2,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
...,...,...,...,...,...,...,...,...,...
95,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,HWP,NaN,238,HWP 표 레코드,7,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
96,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,HWP,NaN,116,HWP 표 레코드,23,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
97,한국한의학연구원_통합정보시스템 고도화 용역.hwp,HWP,NaN,140,HWP 표 레코드,5,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검
98,한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp,HWP,NaN,77,HWP 표 레코드,6,NaN,표 추출·이미지 OCR 검토 필요,빈 페이지 기준 미점검


확인한 문서 수: 100개
PDF 수: 4개
HWP 수: 96개
Tesseract 설치 여부: 설치되지 않음


In [7]:
review_candidates = visual_result[
    visual_result['visual_content_recommendation'] != 'not_required'
].copy()

candidate_display = review_candidates[
    ['file_name', 'document_type', 'table_count', 'image_count', 'visual_content_recommendation']
].copy()
candidate_display.columns = ['파일명', '문서 형식', '표 수', '이미지 수', '추가 검토 결과']
candidate_display['문서 형식'] = candidate_display['문서 형식'].map({'pdf': 'PDF', 'hwp': 'HWP'})
candidate_display['추가 검토 결과'] = candidate_display['추가 검토 결과'].map({
    'table_extraction_review_required': '표 추출 검토 필요',
    'image_ocr_review_required': '이미지 OCR 검토 필요',
    'table_and_image_review_required': '표 추출·이미지 OCR 검토 필요',
})

if candidate_display.empty:
    print('표·이미지가 확인된 문서가 없어 추가 추출 검토 후보가 없습니다.')
else:
    display(candidate_display.sort_values(['이미지 수', '표 수'], ascending=False))

pdf_ocr_candidates = visual_result[
    (visual_result['document_type'] == 'pdf')
    & (visual_result['ocr_recommendation'] != 'not_required')
]

print(f'표·이미지 추가 검토 후보: {len(candidate_display)}개')
print(f'텍스트 미추출 기준 OCR 검토 PDF: {len(pdf_ocr_candidates)}개')
print('결론: 표·이미지 수는 추가 추출 검토 후보를 고르는 지표입니다. OCR 적용 여부는 이미지 안 글자 포함 여부를 대표 문서에서 확인한 뒤 결정합니다.')

,파일명,문서 형식,표 수,이미지 수,추가 검토 결과
12,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,PDF,410,333,표 추출·이미지 OCR 검토 필요
6,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,HWP,184,85,표 추출·이미지 OCR 검토 필요
41,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,PDF,163,43,표 추출·이미지 OCR 검토 필요
96,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,HWP,116,23,표 추출·이미지 OCR 검토 필요
25,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,PDF,63,21,표 추출·이미지 OCR 검토 필요
...,...,...,...,...,...
60,조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp,HWP,91,1,표 추출·이미지 OCR 검토 필요
93,한국지식재산보호원_IP-NAVI 해외지식재산센터 사업관리 시스템 기능개.hwp,HWP,89,1,표 추출·이미지 OCR 검토 필요
77,한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp,HWP,66,1,표 추출·이미지 OCR 검토 필요
84,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,HWP,12,1,표 추출·이미지 OCR 검토 필요


표·이미지 추가 검토 후보: 100개
텍스트 미추출 기준 OCR 검토 PDF: 0개
결론: 표·이미지 수는 추가 추출 검토 후보를 고르는 지표입니다. OCR 적용 여부는 이미지 안 글자 포함 여부를 대표 문서에서 확인한 뒤 결정합니다.
